# 04 - Enthalpy of Formation from NASA Polynomials

The **standard enthalpy of formation** $\Delta_f H^\circ$ is the enthalpy change
when one mole of a species is formed from its elements in their reference states
at 298.15 K and 1 bar. By definition it is **zero** for an element in its
reference state (O₂, N₂, H₂, graphite, ...).

`pyglenn` ships a `calculate_formation_enthalpy` method — but as we will see the
dedicated database column is empty. The good news: because NASA polynomials fit
the *standardized* enthalpy, we can recover $\Delta_f H^\circ$ directly as

$$\Delta_f H^\circ(\text{species}) = H^\circ(298.15\,\mathrm{K}),$$

i.e. simply `calculate_properties(id, 298.15)["h_relative"]`.

In [ ]:
from pyglenn import ThermochemicalCalculator, R

_INDEX = {}

def species_id(calc, name):
    """Return the database id of the species whose *name* matches exactly.

    ``get_available_species`` matches substrings of both the name and the
    formula and caps its result at 20 rows, so short names such as ``"O2"`` can
    be crowded out by entries like ``"AL2O2"`` or ``"CO2"``. To be robust we
    build a full name -> id index once (cached across the session) and look up
    the exact name in it.
    """
    if not _INDEX:
        _INDEX.update({s["name"]: s["id"] for s in calc.get_available_species("")})
    if name not in _INDEX:
        raise ValueError(f"Species {name!r} not found in the database")
    return _INDEX[name]

print("Universal gas constant R =", R, "J/(mol.K)")


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")


## 1. The dedicated method returns `None`

In the bundled database the `heat_of_formation_298K` column was never populated,
so `calculate_formation_enthalpy` returns `None` for every species. It is wired
up and ready should a future database fill that column.

In [ ]:
with ThermochemicalCalculator() as calc:
    for name in ["H2O", "CO2", "CH4"]:
        sid = species_id(calc, name)
        print(f"{name:5s} calculate_formation_enthalpy -> "
              f"{calc.calculate_formation_enthalpy(sid)}")

## 2. Recovering $\Delta_f H^\circ$ from the polynomial

The standardized enthalpy already contains the formation enthalpy, so at
298.15 K it *is* $\Delta_f H^\circ$. Reference-state elements come out at
essentially zero (to within rounding of the fit).

In [ ]:
def formation_enthalpy(calc, name, T=298.15):
    """Delta_f H at T (default 298.15 K), in J/mol, from the standardized H."""
    return calc.calculate_properties(species_id(calc, name), T)["h_relative"]

with ThermochemicalCalculator() as calc:
    print("Reference-state elements (should be ~0):")
    for el in ["O2", "N2", "H2", "C(gr)"]:
        print(f"  {el:6s} {formation_enthalpy(calc, el)/1000: .4f} kJ/mol")

## 3. A validated table of formation enthalpies

We compute $\Delta_f H^\circ$ for a set of common species and compare against
accepted literature values (CODATA / JANAF, gas phase). The agreement is well
within 1 kJ/mol — the polynomials reproduce the reference data they were fit
to.

In [ ]:
# Literature standard enthalpies of formation at 298.15 K, kJ/mol (gas phase)
REFERENCE = {
    "H2": 0.0, "O2": 0.0, "N2": 0.0, "C(gr)": 0.0,
    "H2O": -241.83, "CO2": -393.52, "CO": -110.53, "CH4": -74.87,
    "NH3": -45.90, "NO": 91.29, "NO2": 33.10, "C2H5OH": -235.0,
}

records = []
with ThermochemicalCalculator() as calc:
    for name, ref in REFERENCE.items():
        sid = species_id(calc, name)
        M = calc.db.get_species_data(sid)["molecular_weight"]
        dhf = formation_enthalpy(calc, name) / 1000.0
        records.append({
            "species": name, "M [g/mol]": M,
            "pyglenn [kJ/mol]": dhf, "reference [kJ/mol]": ref,
            "abs.err [kJ/mol]": abs(dhf - ref),
        })

hf_df = pd.DataFrame(records).set_index("species")
print(hf_df.to_string())
print(f"\nLargest absolute error: {hf_df['abs.err [kJ/mol]'].max():.3f} kJ/mol")

## 4. Visualising formation enthalpies

A horizontal bar chart makes the exothermic (negative $\Delta_f H^\circ$, stable)
versus endothermic (positive, energetic) split obvious. Elements sit exactly on
the zero line.

In [ ]:
order = hf_df.sort_values("pyglenn [kJ/mol]")
vals = order["pyglenn [kJ/mol]"]
colors = ["#c0392b" if v > 0 else ("#7f8c8d" if abs(v) < 1e-3 else "#2471a3")
          for v in vals]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(order.index, vals, color=colors)
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel(r"$\Delta_f H^\circ$ at 298.15 K [kJ/mol]")
ax.set_title("Standard enthalpies of formation (from pyglenn)")
plt.show()

## 5. Only 298.15 K is "formation"

$\Delta_f H^\circ$ is defined at 298.15 K. The standardized enthalpy $H^\circ(T)$
keeps varying with temperature, but only its 298.15 K value equals the tabulated
formation enthalpy. For CO₂ we plot $H^\circ(T)$ and highlight that reference
point; the vertical offset above it is the *sensible* enthalpy
$H^\circ(T)-H^\circ(298.15)$ from notebook 03.

In [ ]:
Tgrid = np.linspace(300, 2500, 60)
with ThermochemicalCalculator() as calc:
    co2 = species_id(calc, "CO2")
    H = np.array([calc.calculate_properties(co2, t)["h_relative"] for t in Tgrid]) / 1000.0
    hf298 = formation_enthalpy(calc, "CO2") / 1000.0

fig, ax = plt.subplots()
ax.plot(Tgrid, H, label=r"$H^\circ(T)$ of CO$_2$")
ax.scatter([298.15], [hf298], color="red", zorder=5,
           label=fr"$\Delta_f H^\circ$ = {hf298:.1f} kJ/mol at 298.15 K")
ax.set_xlabel("Temperature [K]")
ax.set_ylabel("Standardized enthalpy [kJ/mol]")
ax.set_title("Formation enthalpy is the 298.15 K value of the standardized enthalpy")
ax.legend()
plt.show()

## Summary

- `calculate_formation_enthalpy` returns `None` here (empty database column).
- Use `calculate_properties(id, 298.15)["h_relative"]` instead — it equals
  $\Delta_f H^\circ$ and reproduces literature values to < 1 kJ/mol.
- Because $H^\circ$ already carries formation enthalpy, reaction enthalpies
  become trivial sums.

**Next:** notebook 05 uses exactly this to compute reaction enthalpies and heats
of combustion.